# Laboratorio 4 — Análisis de Datos GeoEspaciales
### CC3084 — Data Science | UVG | Semestre II - 2026

**Ejercicios 1 y 2**

1. Establecer conexión con el API de Sentinel-2 usando el módulo `openeo`.
2. Obtener únicamente los datos raster necesarios (bandas B03, B04, B08) para el análisis
   de los lagos Atitlán y Amatitlán, usando las coordenadas provistas en el enunciado y
   exclusivamente las fechas oficiales indicadas en el laboratorio.

> El índice de cianobacteria (Cyano Detection Script) se trabaja en el Ejercicio 3 usando
> Sentinel Hub / Copernicus Browser directamente, o reproduciéndolo localmente con las
> bandas mínimas necesarias. Este notebook se enfoca en dejar lista la conexión y la
> descarga de las bandas base (B03, B04, B08) que alimentan NDVI y NDWI, y que sirven
> como insumo si se decide reproducir el índice de cianobacteria localmente.

## Ejercicio 1. Conexión con el API de Sentinel-2 (openEO)

Usamos el backend openEO de la **Copernicus Data Space Ecosystem (CDSE)**, que es el
punto de acceso oficial (gratuito, con cuenta Copernicus) a las colecciones de
Sentinel-2 L2A. La autenticación es OIDC (abre una ventana/URL de login la primera vez
y luego cachea el token localmente).

In [ ]:
# Si no lo tienes instalado:
# %pip install openeo geopandas shapely matplotlib pandas --quiet

import openeo
from pathlib import Path
import pandas as pd

# --- Conexión al backend openEO de Copernicus Data Space Ecosystem ---
OPENEO_URL = "https://openeo.dataspace.copernicus.eu"

connection = openeo.connect(OPENEO_URL)

# Autenticación OIDC: la primera vez abrirá una URL/navegador para iniciar sesión
# con tu usuario y contraseña de Copernicus. El token queda cacheado para las
# siguientes ejecuciones (no es necesario volver a loguearse cada vez).
connection.authenticate_oidc()

print("Conectado a:", OPENEO_URL)
print("Usuario autenticado correctamente.")

In [ ]:
# Verificamos que la colección Sentinel-2 L2A esté disponible en el backend
colecciones = connection.list_collection_ids()
assert "SENTINEL2_L2A" in colecciones, "La colección SENTINEL2_L2A no está disponible en este backend"
print("Colección SENTINEL2_L2A disponible. Listo para continuar.")

## Ejercicio 2. Obtención de los datos raster necesarios

Para reducir tiempos de descarga y almacenamiento, **solo se descargan las bandas
necesarias para calcular NDVI y NDWI**:

- **NDVI:** `B04` (rojo) y `B08` (infrarrojo cercano)
- **NDWI:** `B03` (verde) y `B08` (infrarrojo cercano)

Es decir, con las tres bandas `B03`, `B04`, `B08` cubrimos ambos índices en una sola
descarga por imagen (evitamos descargar la escena completa).

Usamos exclusivamente:
- Las coordenadas (bounding box) de cada lago dadas en el enunciado.
- Las fechas oficiales de la tabla del laboratorio (11 fechas por lago).

In [ ]:
# --- Coordenadas de cada lago (dadas en el enunciado del laboratorio) ---
lagos_bbox = {
    "atitlan": {
        "west": -91.326256,
        "east": -91.07151,
        "south": 14.5948,
        "north": 14.750979,
    },
    "amatitlan": {
        "west": -90.638065,
        "east": -90.512924,
        "south": 14.412347,
        "north": 14.493799,
    },
}

# --- Fechas oficiales por lago (tal como las proporciona el laboratorio) ---
fechas_atitlan = [
    "2025-01-18", "2025-04-13", "2025-05-13", "2025-07-17", "2025-11-21",
    "2025-12-29", "2026-02-12", "2026-03-24", "2026-04-13", "2026-04-28",
    "2026-07-22",
]

fechas_amatitlan = [
    "2025-01-28", "2025-04-15", "2025-04-28", "2025-11-24", "2026-01-08",
    "2026-02-02", "2026-02-07", "2026-03-29", "2026-04-13", "2026-04-28",
    "2026-06-19",
]

fechas_por_lago = {
    "atitlan": fechas_atitlan,
    "amatitlan": fechas_amatitlan,
}

for lago, fechas in fechas_por_lago.items():
    print(f"{lago}: {len(fechas)} fechas oficiales")

In [ ]:
# --- Bandas mínimas necesarias: B03 (verde), B04 (rojo), B08 (NIR) ---
BANDAS = ["B03", "B04", "B08"]

# Carpeta de salida para los raster descargados
OUTPUT_DIR = Path("data/raster")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def siguiente_dia(fecha_str: str) -> str:
    """Devuelve la fecha siguiente en formato YYYY-MM-DD (para construir un
    rango temporal de un solo día, requerido por openEO)."""
    fecha = pd.to_datetime(fecha_str)
    return (fecha + pd.Timedelta(days=1)).strftime("%Y-%m-%d")


def descargar_imagen(lago: str, fecha: str, bbox: dict, bandas=BANDAS,
                      max_cloud_cover: float = 20.0) -> Path:
    """Descarga únicamente las bandas necesarias para una fecha y un lago
    dados, usando el bounding box provisto. Guarda el resultado como GeoTIFF.
    """
    destino = OUTPUT_DIR / lago / f"{lago}_{fecha}.tif"
    destino.parent.mkdir(parents=True, exist_ok=True)

    if destino.exists():
        print(f"[omitido] Ya existe: {destino}")
        return destino

    datacube = connection.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=bbox,
        temporal_extent=[fecha, siguiente_dia(fecha)],
        bands=bandas,
        max_cloud_cover=max_cloud_cover,
    )

    # Nos quedamos con la mediana temporal por si hay más de una escena en
    # el rango (debería ser una sola, dado que el rango es de un día)
    datacube = datacube.reduce_dimension(dimension="t", reducer="median")

    print(f"Descargando {lago} - {fecha} ...")
    datacube.download(str(destino))
    print(f"  -> guardado en {destino}")
    return destino

In [ ]:
# --- Descarga de todas las imágenes (solo bandas necesarias) para ambos lagos ---
rutas_descargadas = []

for lago, bbox in lagos_bbox.items():
    for fecha in fechas_por_lago[lago]:
        ruta = descargar_imagen(lago, fecha, bbox)
        rutas_descargadas.append({"lago": lago, "fecha": fecha, "ruta": str(ruta)})

df_descargas = pd.DataFrame(rutas_descargadas)
df_descargas

In [ ]:
# Guardamos el índice de archivos descargados para usarlo en los siguientes
# ejercicios (cálculo de índices, análisis temporal y espacial)
df_descargas.to_csv("data/indice_descargas.csv", index=False)
print("Total de imágenes descargadas:", len(df_descargas))
df_descargas.groupby("lago").size()